In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, List, Tuple
from dataclasses import dataclass

@dataclass
class ParameterAnalysis:
    name: str
    best_value: float
    avg_f1_macro: float
    avg_f1_weighted: float
    min_f1_macro: float
    min_f1_weighted: float
    max_f1_macro: float
    max_f1_weighted: float
    std_f1_macro: float
    std_f1_weighted: float
    count: int

class HyperparameterAnalyzer:
    def __init__(self, data_path: str):
        self.df = pd.read_csv(data_path)
        self.parameters = [
            'model_dim', 'num_encoder_layers', 'num_heads', 'batch_size',
            'learning_rate', 'dropout_rate', 'activation', 'norm_first',
            'pooling_type', 'focal_gamma', 'optimizer', 
            'lr_scheduler', 'weight_decay', 'label_smoothing', 'l1_lambda'
        ]
        
        self.parameter_name_map = {
            'learning_rate': 'Learning Rate',
            'dropout_rate': 'Dropout Rate',
            'batch_size': 'Batch Größe',
            'optimizer': 'Optimizer',
            'model_dim': 'Modell Dimension',
            'num_encoder_layers': 'Anzahl der Encoder Schichten',
            'num_heads': 'Anzahl der Attention Köpfe',
            'activation': 'Activierungsfunktion',
            'label_smoothing': 'Label-Smoothing',
            'pooling_type': 'Pooling-Typ',
            'norm_first': 'Norm First',
            'l1_lambda': 'L1-Regularisierung (l1_lambda)',
            'weight_decay': 'L2-Regularisierung (weight_decay)',
            'lr_scheduler': 'Learning Rate Scheduler',
            'focal_gamma': 'Focal Cross Entropy Loss γ'
        }
        
        self.log_scale_params = {'learning_rate', 'weight_decay', 'l1_lambda', 'focal_gamma'}

    def analyze_parameter(self, param: str) -> ParameterAnalysis:
        grouped = self.df.groupby(param)[['val/f1_score_macro', 'val/f1_score_weighted']].agg([
            'mean', 'min', 'max', 'std', 'count'
        ]).reset_index()
        
        best_idx = grouped[('val/f1_score_weighted', 'mean')].idxmax()
        best_config = grouped.iloc[best_idx]
        
        return ParameterAnalysis(
            name=param,
            best_value=best_config[param],
            avg_f1_macro=best_config[('val/f1_score_macro', 'mean')],
            avg_f1_weighted=best_config[('val/f1_score_weighted', 'mean')],
            min_f1_macro=best_config[('val/f1_score_macro', 'min')],
            min_f1_weighted=best_config[('val/f1_score_weighted', 'min')],
            max_f1_macro=best_config[('val/f1_score_macro', 'max')],
            max_f1_weighted=best_config[('val/f1_score_weighted', 'max')],
            std_f1_macro=best_config[('val/f1_score_macro', 'std')],
            std_f1_weighted=best_config[('val/f1_score_weighted', 'std')],
            count=best_config[('val/f1_score_macro', 'count')]
        )

    def plot_parameter_influence(self, param: str, save_path: str = None):
        """Erstellt ein Boxplot-Diagramm für den Einfluss eines Parameters."""
        # Erstelle Figure mit extra Platz für die Legende rechts
        plt.figure(figsize=(12, 6))  # Breite erhöht für Platz der Legende
        
        # Erstelle ein Subplot mit definiertem Bereich für das Diagramm
        ax = plt.subplot(111)
        
        # Daten für das Boxplot vorbereiten
        data_macro = []
        data_weighted = []
        labels = []
        counts = []
        
        for value in sorted(self.df[param].unique()):
            subset = self.df[self.df[param] == value]
            counts.append(len(subset))
            
            macro_values = subset['val/f1_score_macro'].values * 100
            weighted_values = subset['val/f1_score_weighted'].values * 100
            
            # NaN-Werte entfernen
            macro_values = macro_values[~np.isnan(macro_values)]
            weighted_values = weighted_values[~np.isnan(weighted_values)]
            
            data_macro.append(macro_values)
            data_weighted.append(weighted_values)
            labels.append(str(value))

        positions = np.arange(len(labels)) * 2
        width = 0.8
        bp1 = ax.boxplot(data_macro, positions=positions-width/2, widths=width,
                        patch_artist=True, labels=[''] * len(labels))
        bp2 = ax.boxplot(data_weighted, positions=positions+width/2, widths=width,
                        patch_artist=True, labels=labels)

        ax.set_ylim(0, 100)
        
        # Anzahl der Experimente unter den Boxplots anzeigen
        for i, count in enumerate(counts):
            ax.text(positions[i], -8, f'n={count}', 
                    horizontalalignment='center', verticalalignment='top')

        plt.setp(bp1['boxes'], facecolor='lightblue', alpha=0.7)
        plt.setp(bp2['boxes'], facecolor='lightgreen', alpha=0.7)
        
        param_titles = {
            'learning_rate': 'Learning Rate',
            'dropout_rate': 'Dropout Rate',
            'batch_size': 'Batch Größe',
            'optimizer': 'Optimizer',
            'model_dim': 'Modell Dimension',
            'num_encoder_layers': 'Anzahl der Encoder Schichten',
            'num_heads': 'Anzahl der Attention Köpfe',
            'activation': 'Aktivierungsfunktion',
            'label_smoothing': 'Label-Smoothing',
            'pooling_type': 'Pooling-Typ',
            'norm_first': 'Norm First',
            'l1_lambda': 'L1-Regularisierung',
            'weight_decay': 'L2-Regularisierung',
            'lr_scheduler': 'Learning Rate Scheduler',
            'focal_gamma': 'Focal Cross Entropy Loss γ'
        }
        
        ax.set_title(f'Einfluss von {param_titles.get(param, param)} auf F1-Scores')
        ax.set_xlabel(param_titles.get(param, param), labelpad=20)
        ax.set_ylabel('F1-Score (%)')
        
        ax.set_xticks(positions)
        ax.set_xticklabels(labels, rotation=45 if len(max(labels, key=len)) > 6 else 0)
        
        # Legende außerhalb des Plots platzieren
        ax.plot([], [], 'lightblue', label='Macro F1')
        ax.plot([], [], 'lightgreen', label='Weighted F1')
        ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        
        ax.grid(True, alpha=0.3)
        
        # Adjustiere Layout um sicherzustellen, dass alles sichtbar ist
        plt.tight_layout()
        
        if save_path:
            plt.savefig(f'{save_path}/{param}_analysis.png', 
                    bbox_inches='tight', 
                    dpi=300,
                    pad_inches=0.1)
        plt.close()

    def create_analysis_table(self) -> pd.DataFrame:
        results = []
        for param in self.parameters:
            analysis = self.analyze_parameter(param)
            display_name = self.parameter_name_map.get(param, param)
            results.append({
                'Parameter': display_name,
                'Bester Wert': analysis.best_value,
                'Durchschn. F1 Macro': f'{analysis.avg_f1_macro*100:.2f}%',
                'Durchschn. F1 Weighted': f'{analysis.avg_f1_weighted*100:.2f}%',
                'Min F1 Macro': f'{analysis.min_f1_macro*100:.2f}%',
                'Min F1 Weighted': f'{analysis.min_f1_weighted*100:.2f}%',
                'Max F1 Macro': f'{analysis.max_f1_macro*100:.2f}%',
                'Max F1 Weighted': f'{analysis.max_f1_weighted*100:.2f}%',
                'Std.abw. Macro': f'{analysis.std_f1_macro*100:.2f}%',
                'Std.abw. Weighted': f'{analysis.std_f1_weighted*100:.2f}%',
                'n': analysis.count
            })
        return pd.DataFrame(results)

    def analyze_all(self, output_dir: str):
        import os
        os.makedirs(output_dir, exist_ok=True)
        
        for param in self.parameters:
            print(f"\nAnalysiere {param}:")
            analysis = self.analyze_parameter(param)
            print(f"Bester Wert: {analysis.best_value}")
            print(f"Durchschn. F1 Macro: {analysis.avg_f1_macro*100:.2f}%")
            print(f"Durchschn. F1 Weighted: {analysis.avg_f1_weighted*100:.2f}%")
            print(f"Std Macro: {analysis.std_f1_macro*100:.2f}%")
            print(f"Std Weighted: {analysis.std_f1_weighted*100:.2f}%")
            print(f"Anzahl Experimente: {analysis.count}")
            
            self.plot_parameter_influence(param, output_dir)
        
        df = self.create_analysis_table()
        df.to_csv(f'{output_dir}/hyperparameter_analysis.csv', index=False)

# Analyse durchführen
analyzer = HyperparameterAnalyzer('src/evaluation/wandb_export_200.csv')
analyzer.analyze_all('hyperparameter_analysis_results/200')


Analysiere model_dim:
Bester Wert:     256.0
Name: 1, dtype: float64
Durchschn. F1 Macro: 50.92%
Durchschn. F1 Weighted: 84.57%
Std Macro: 30.78%
Std Weighted: 21.48%
Anzahl Experimente: 21.0


/tmp/ipykernel_162351/3400622800.py:105: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp1 = ax.boxplot(data_macro, positions=positions-width/2, widths=width,
/tmp/ipykernel_162351/3400622800.py:107: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp2 = ax.boxplot(data_weighted, positions=positions+width/2, widths=width,



Analysiere num_encoder_layers:
Bester Wert:     6.0
Name: 4, dtype: float64
Durchschn. F1 Macro: 58.98%
Durchschn. F1 Weighted: 94.91%
Std Macro: 17.93%
Std Weighted: 3.02%
Anzahl Experimente: 10.0


/tmp/ipykernel_162351/3400622800.py:105: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp1 = ax.boxplot(data_macro, positions=positions-width/2, widths=width,
/tmp/ipykernel_162351/3400622800.py:107: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp2 = ax.boxplot(data_weighted, positions=positions+width/2, widths=width,



Analysiere num_heads:
Bester Wert:     4.0
Name: 1, dtype: float64
Durchschn. F1 Macro: 61.82%
Durchschn. F1 Weighted: 93.50%
Std Macro: 22.87%
Std Weighted: 7.27%
Anzahl Experimente: 23.0


...

/tmp/ipykernel_162351/3400622800.py:105: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp1 = ax.boxplot(data_macro, positions=positions-width/2, widths=width,
/tmp/ipykernel_162351/3400622800.py:107: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp2 = ax.boxplot(data_weighted, positions=positions+width/2, widths=width,



Analysiere label_smoothing:
Bester Wert:     0.05
Name: 2, dtype: float64
Durchschn. F1 Macro: 63.93%
Durchschn. F1 Weighted: 92.78%
Std Macro: 21.78%
Std Weighted: 13.40%
Anzahl Experimente: 15.0


/tmp/ipykernel_162351/3400622800.py:105: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp1 = ax.boxplot(data_macro, positions=positions-width/2, widths=width,
/tmp/ipykernel_162351/3400622800.py:107: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp2 = ax.boxplot(data_weighted, positions=positions+width/2, widths=width,



Analysiere l1_lambda:
Bester Wert:     0.0001
Name: 2, dtype: float64
Durchschn. F1 Macro: 47.58%
Durchschn. F1 Weighted: 83.21%
Std Macro: 24.25%
Std Weighted: 26.45%
Anzahl Experimente: 32.0


/tmp/ipykernel_162351/3400622800.py:105: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp1 = ax.boxplot(data_macro, positions=positions-width/2, widths=width,
/tmp/ipykernel_162351/3400622800.py:107: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp2 = ax.boxplot(data_weighted, positions=positions+width/2, widths=width,
